# Advanced Problems with Solutions: Integer Operations
Focus: exact integer arithmetic, floor division, modulo, `divmod`, negative operands, exponentiation, and avoiding float-based mistakes.

In [1]:
import math
import random
import sys
from timeit import repeat

print(sys.version)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


## Problem 1 — Operation result types

Show which integer operations preserve `int` and which produce `float`.

In [2]:
expressions = {
    "2 + 3": 2 + 3,
    "2 - 3": 2 - 3,
    "2 * 3": 2 * 3,
    "2 ** 3": 2 ** 3,
    "2 ** -3": 2 ** -3,
    "10 / 2": 10 / 2,
    "10 // 2": 10 // 2,
    "10 % 2": 10 % 2,
}

for expr, value in expressions.items():
    print(f"{expr:8} -> {value!r:8} {type(value).__name__}")

2 + 3    -> 5        int
2 - 3    -> -1       int
2 * 3    -> 6        int
2 ** 3   -> 8        int
2 ** -3  -> 0.125    float
10 / 2   -> 5.0      float
10 // 2  -> 5        int
10 % 2   -> 0        int


## Problem 2 — Verify the division algorithm

For many random integers `a` and nonzero integers `b`, verify:

`a == b * (a // b) + (a % b)`

In [3]:
rng = random.Random(42)

for _ in range(10_000):
    a = rng.randint(-10**12, 10**12)
    b = rng.randint(-10**6, 10**6)
    if b == 0:
        continue

    q = a // b
    r = a % b
    assert a == b * q + r

print("All randomized tests passed.")

All randomized tests passed.


## Problem 3 — Sign of the remainder

Investigate the sign rule for `a % b`.

In [4]:
cases = [(13, 4), (-13, 4), (13, -4), (-13, -4)]

for a, b in cases:
    q, r = divmod(a, b)
    print(f"a={a:>4}, b={b:>4} | a//b={q:>4}, a%b={r:>4}, check={a == b*q + r}")

print("\nObservation: when r != 0, Python's remainder has the same sign as b.")

a=  13, b=   4 | a//b=   3, a%b=   1, check=True
a= -13, b=   4 | a//b=  -4, a%b=   3, check=True
a=  13, b=  -4 | a//b=  -4, a%b=  -3, check=True
a= -13, b=  -4 | a//b=   3, a%b=  -1, check=True

Observation: when r != 0, Python's remainder has the same sign as b.


## Problem 4 — Implement truncating division

Python's `//` floors. Some languages truncate toward zero. Implement truncating quotient and remainder exactly, without using `/`.

In [5]:
def trunc_divmod(a: int, b: int) -> tuple[int, int]:
    if b == 0:
        raise ZeroDivisionError("division by zero")

    q = abs(a) // abs(b)
    if (a < 0) != (b < 0):
        q = -q
    r = a - b * q
    return q, r

for a, b in [(13, 4), (-13, 4), (13, -4), (-13, -4)]:
    print(f"{a:>4}, {b:>4} -> floor={divmod(a,b)}, trunc={trunc_divmod(a,b)}")

for _ in range(10_000):
    a = rng.randint(-10**9, 10**9)
    b = rng.randint(-10**6, 10**6) or 1
    q, r = trunc_divmod(a, b)
    assert a == b * q + r
    assert abs(r) < abs(b)

print("Truncating division tests passed.")

  13,    4 -> floor=(3, 1), trunc=(3, 1)
 -13,    4 -> floor=(-4, 3), trunc=(-3, -1)
  13,   -4 -> floor=(-4, -3), trunc=(-3, 1)
 -13,   -4 -> floor=(3, -1), trunc=(3, -1)
Truncating division tests passed.


## Problem 5 — Never use float division for huge integer floor division

Show why `math.floor(a / b)` is unsafe for very large integers.

In [6]:
a = 10**100 + 123456789
b = 97

correct = a // b

try:
    unsafe = math.floor(a / b)
    print("unsafe == correct:", unsafe == correct)
except OverflowError as exc:
    print("Float conversion failed:", exc)

print("Correct result computed exactly with //.")
print("quotient bit length:", correct.bit_length())

unsafe == correct: False
Correct result computed exactly with //.
quotient bit length: 326


## Problem 6 — Use `divmod` efficiently

Write a function that converts a non-negative integer into digits in an arbitrary base using `divmod`.

In [7]:
def digits_in_base(n: int, base: int) -> list[int]:
    if n < 0:
        raise ValueError("n must be non-negative")
    if base < 2:
        raise ValueError("base must be at least 2")
    if n == 0:
        return [0]

    digits = []
    while n:
        n, remainder = divmod(n, base)
        digits.append(remainder)

    return digits[::-1]

print(digits_in_base(255, 2))
print(digits_in_base(255, 16))
print(digits_in_base(123456789, 10))

assert digits_in_base(255, 16) == [15, 15]
assert digits_in_base(0, 10) == [0]

[1, 1, 1, 1, 1, 1, 1, 1]
[15, 15]
[1, 2, 3, 4, 5, 6, 7, 8, 9]


## Problem 7 — Modular arithmetic normalization

Create `normalize_mod(a, m)` returning the canonical representative in `[0, m)`.

In [8]:
def normalize_mod(a: int, m: int) -> int:
    if m <= 0:
        raise ValueError("m must be positive")
    return a % m

for a in [-21, -20, -19, -1, 0, 1, 19, 20, 21]:
    print(f"{a:>4} mod 5 -> {normalize_mod(a, 5)}")

for _ in range(10_000):
    a = rng.randint(-10**9, 10**9)
    m = rng.randint(1, 10**6)
    r = normalize_mod(a, m)
    assert 0 <= r < m
    assert (a - r) % m == 0

print("Normalization tests passed.")

 -21 mod 5 -> 4
 -20 mod 5 -> 0
 -19 mod 5 -> 1
  -1 mod 5 -> 4
   0 mod 5 -> 0
   1 mod 5 -> 1
  19 mod 5 -> 4
  20 mod 5 -> 0
  21 mod 5 -> 1
Normalization tests passed.


## Problem 8 — Modular inverse

Use Python's built-in modular inverse support: `pow(a, -1, m)`.

In [9]:
def mod_inverse(a: int, m: int) -> int:
    if m <= 0:
        raise ValueError("m must be positive")
    return pow(a, -1, m)

for a, m in [(3, 11), (10, 17), (1234567, 1_000_000_007)]:
    inv = mod_inverse(a, m)
    print(f"inverse of {a} mod {m} = {inv}")
    assert (a * inv) % m == 1

try:
    mod_inverse(6, 15)
except ValueError as exc:
    print("No inverse exists:", exc)

inverse of 3 mod 11 = 4
inverse of 10 mod 17 = 12
inverse of 1234567 mod 1000000007 = 989145189
No inverse exists: base is not invertible for the given modulus


## Problem 9 — Fast modular exponentiation

Compare `(a ** b) % m` with `pow(a, b, m)`.

In [10]:
a = 987654321987654321
b = 20_000
m = 1_000_000_007

assert (a ** b) % m == pow(a, b, m)

slow = min(repeat("(a ** b) % m", globals=globals(), number=1, repeat=3))
fast = min(repeat("pow(a, b, m)", globals=globals(), number=1, repeat=5))

print("slow seconds:", slow)
print("fast seconds:", fast)
print("speedup:", slow / fast)

slow seconds: 0.06422100029885769
fast seconds: 8.00006091594696e-07
speedup: 80275.63911525029


## Problem 10 — Last `k` digits of a huge power

Compute the last `k` decimal digits of `a ** b` without constructing the full power.

In [11]:
def last_digits(a: int, b: int, k: int) -> str:
    if b < 0:
        raise ValueError("b must be non-negative")
    if k < 1:
        raise ValueError("k must be positive")

    result = pow(a, b, 10**k)
    return f"{result:0{k}d}"

print(last_digits(2, 100, 10))
print(last_digits(123456789, 987654321, 20))

6703205376
64922883132974933589


## Problem 11 — Exact divisibility tests

Implement common divisibility checks using `%`.

In [12]:
def is_even(n: int) -> bool:
    return n % 2 == 0

def is_divisible_by(n: int, d: int) -> bool:
    if d == 0:
        raise ZeroDivisionError("division by zero")
    return n % d == 0

for n in [-10, -9, 0, 9, 10]:
    print(f"{n:>3}: even={is_even(n)}, divisible by 3={is_divisible_by(n, 3)}")

assert is_divisible_by(10**100 - 1, 9)
assert not is_divisible_by(10**100 + 1, 9)

-10: even=True, divisible by 3=False
 -9: even=False, divisible by 3=True
  0: even=True, divisible by 3=True
  9: even=False, divisible by 3=True
 10: even=True, divisible by 3=False


## Problem 12 — Chunking with floor division and modulo

Given a byte count, compute full chunks and leftover bytes.

In [13]:
def chunk_info(total_bytes: int, chunk_size: int) -> dict[str, int]:
    if total_bytes < 0:
        raise ValueError("total_bytes must be non-negative")
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    full_chunks, leftover = divmod(total_bytes, chunk_size)
    needed_chunks = full_chunks + (leftover != 0)

    return {
        "full_chunks": full_chunks,
        "leftover": leftover,
        "needed_chunks": needed_chunks,
    }

for total in [0, 1, 1023, 1024, 1025, 10_000]:
    print(total, "->", chunk_info(total, 1024))

0 -> {'full_chunks': 0, 'leftover': 0, 'needed_chunks': 0}
1 -> {'full_chunks': 0, 'leftover': 1, 'needed_chunks': 1}
1023 -> {'full_chunks': 0, 'leftover': 1023, 'needed_chunks': 1}
1024 -> {'full_chunks': 1, 'leftover': 0, 'needed_chunks': 1}
1025 -> {'full_chunks': 1, 'leftover': 1, 'needed_chunks': 2}
10000 -> {'full_chunks': 9, 'leftover': 784, 'needed_chunks': 10}


## Problem 13 — Ceiling division

Implement exact ceiling division for integers.

In [14]:
def ceil_div(a: int, b: int) -> int:
    if b == 0:
        raise ZeroDivisionError("division by zero")
    return -((-a) // b)

for a, b in [(13, 4), (-13, 4), (13, -4), (-13, -4)]:
    print(f"ceil_div({a}, {b}) = {ceil_div(a, b)}")

for _ in range(10_000):
    a = rng.randint(-10**9, 10**9)
    b = rng.randint(-10**6, 10**6) or 1
    assert ceil_div(a, b) == math.ceil(a / b) or abs(a) > 2**53

print("Exact ceiling division implemented without relying on float arithmetic.")

ceil_div(13, 4) = 4
ceil_div(-13, 4) = -3
ceil_div(13, -4) = -3
ceil_div(-13, -4) = 4
Exact ceiling division implemented without relying on float arithmetic.


## Problem 14 — Euclidean GCD from modulo

Implement `gcd` using repeated modulo.

In [15]:
def gcd_euclid(a: int, b: int) -> int:
    a, b = abs(a), abs(b)
    while b:
        a, b = b, a % b
    return a

test_cases = [(54, 24), (-54, 24), (0, 5), (5, 0), (0, 0), (10**100 + 1, 10**50 + 1)]

for a, b in test_cases:
    print(f"gcd({a}, {b}) = {gcd_euclid(a, b)}")
    assert gcd_euclid(a, b) == math.gcd(a, b)

gcd(54, 24) = 6
gcd(-54, 24) = 6
gcd(0, 5) = 5
gcd(5, 0) = 5
gcd(0, 0) = 0
gcd(10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001, 100000000000000000000000000000000000000000000000001) = 1


## Problem 15 — Benchmark `divmod` versus separate operations

Compare computing quotient and remainder separately with using `divmod`.

In [16]:
a = 10**80 + 123456789
b = 97

def separate(a: int, b: int) -> tuple[int, int]:
    return a // b, a % b

def together(a: int, b: int) -> tuple[int, int]:
    return divmod(a, b)

assert separate(a, b) == together(a, b)

separate_time = min(repeat("separate(a, b)", globals=globals(), number=100_000, repeat=5))
together_time = min(repeat("together(a, b)", globals=globals(), number=100_000, repeat=5))

print("separate time:", separate_time)
print("divmod time:  ", together_time)
print("ratio separate/divmod:", separate_time / together_time)

separate time: 0.02776119951158762
divmod time:   0.02268880046904087
ratio separate/divmod: 1.2235640024014534


## Summary

- `/` always returns `float`.
- `//` performs floor division, not truncation toward zero.
- `%` pairs with `//` through `a == b * (a // b) + a % b`.
- The remainder has the same sign as the divisor, unless it is zero.
- Use `divmod(a, b)` when you need both quotient and remainder.
- Use `pow(a, b, m)` for modular exponentiation.
- Avoid `math.floor(a / b)` for large integers; use `a // b`.
- Use `-((-a) // b)` for exact ceiling division.